# UCI SECOM Dataset - SVM Classification

In [ ]:
## 1. Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
## 2. Load Dataset

df = pd.read_csv("../data/uci-secom.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (1567, 592)


,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


## 3. Prepare Features and Target

The `Time` column is not used as a machine learning feature because it represents the timestamp of the measurement.

The `Pass/Fail` column is used as the target variable.

- `X` → sensor/measurement features
- `y` → Pass/Fail target

In [ ]:
## 3. Prepare Features and Target

X = df.drop(columns=["Time", "Pass/Fail"])
y = df["Pass/Fail"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1567, 590)
y shape: (1567,)


In [ ]:
## 4. Remove Constant Features

constant_columns = X.columns[X.nunique() <= 1]

print("Constant features found:", len(constant_columns))

X = X.drop(columns=constant_columns)

print("X shape after removing constant features:", X.shape)

Constant features found: 116
X shape after removing constant features: (1567, 474)


In [ ]:
## 5. Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (1253, 474)
X_test : (314, 474)
y_train: (1253,)
y_test : (314,)


## 6. Handle Missing Values

The SECOM dataset contains missing values in several measurement features.

Median imputation is used to replace missing values. The median is calculated from the training data and then applied to the test data.

In [ ]:
## 6. Handle Missing Values

imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("Missing values in X_train:", np.isnan(X_train).sum())
print("Missing values in X_test :", np.isnan(X_test).sum())

X_train shape: (1253, 474)
X_test shape : (314, 474)
Missing values in X_train: 0
Missing values in X_test : 0


## 7. Feature Scaling

SVM is sensitive to the scale of input features.

StandardScaler is used to standardize the features so that they have a comparable scale.

In [ ]:
## 7. Feature Scaling

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (1253, 474)
X_test shape : (314, 474)


## 8. Linear SVM

A linear SVM is tested first to establish a baseline.

The `class_weight="balanced"` option is used because the dataset contains many more Pass samples than Fail samples.

In [ ]:
## 8. Linear SVM

linear_svm = SVC(
    kernel="linear",
    class_weight="balanced"
)

linear_svm.fit(X_train, y_train)

linear_pred = linear_svm.predict(X_test)

print("First 20 predictions:")
print(linear_pred[:20])

First 20 predictions:
[-1  1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  1 -1 -1]


In [ ]:
## 9. Linear SVM Evaluation

print("Linear SVM Accuracy:")
print(accuracy_score(y_test, linear_pred))

print("\nClassification Report:")
print(classification_report(y_test, linear_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, linear_pred))

Linear SVM Accuracy:
0.8535031847133758

Classification Report:
              precision    recall  f1-score   support

          -1       0.94      0.90      0.92       293
           1       0.12      0.19      0.15        21

    accuracy                           0.85       314
   macro avg       0.53      0.55      0.53       314
weighted avg       0.88      0.85      0.87       314


Confusion Matrix:
[[264  29]
 [ 17   4]]


## 10. RBF SVM

An RBF (Radial Basis Function) kernel is tested because the relationship between semiconductor measurements and Pass/Fail may not be linearly separable.

In [ ]:
## 10. RBF SVM

rbf_svm = SVC(
    kernel="rbf",
    class_weight="balanced"
)

rbf_svm.fit(X_train, y_train)

rbf_pred = rbf_svm.predict(X_test)

print("RBF SVM predictions:")
print(rbf_pred[:20])

RBF SVM predictions:
[-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]


In [ ]:
## 11. RBF SVM Evaluation

print("RBF SVM Accuracy:")
print(accuracy_score(y_test, rbf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rbf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rbf_pred))

RBF SVM Accuracy:
0.9235668789808917

Classification Report:
              precision    recall  f1-score   support

          -1       0.94      0.99      0.96       293
           1       0.20      0.05      0.08        21

    accuracy                           0.92       314
   macro avg       0.57      0.52      0.52       314
weighted avg       0.89      0.92      0.90       314


Confusion Matrix:
[[289   4]
 [ 20   1]]


## 12. C Parameter Experiment

The C parameter controls how strongly the SVM penalizes classification errors.

Different C values are tested to observe their effect on model performance.

In [ ]:
## 12. C Parameter Experiment

C_values = [0.1, 1, 10, 100]

for c in C_values:

    model = SVC(
        kernel="rbf",
        C=c,
        class_weight="balanced"
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    print(f"C = {c}")
    print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
    print(f"Fail Precision: {precision_score(y_test, predictions, pos_label=1, zero_division=0):.3f}")
    print(f"Fail Recall: {recall_score(y_test, predictions, pos_label=1, zero_division=0):.3f}")
    print(f"Fail F1: {f1_score(y_test, predictions, pos_label=1, zero_division=0):.3f}")
    print()

C = 0.1
Accuracy: 0.812
Fail Precision: 0.161
Fail Recall: 0.429
Fail F1: 0.234

C = 1
Accuracy: 0.924
Fail Precision: 0.200
Fail Recall: 0.048
Fail F1: 0.077

C = 10
Accuracy: 0.936
Fail Precision: 1.000
Fail Recall: 0.048
Fail F1: 0.091

C = 100
Accuracy: 0.936
Fail Precision: 1.000
Fail Recall: 0.048
Fail F1: 0.091



## 13. Gamma Parameter Experiment

Gamma controls how far the influence of a single training example reaches in an RBF SVM.

Different gamma values are tested to observe their effect on performance.

In [ ]:
## 13. Gamma Parameter Experiment

gamma_values = [0.001, 0.01, 0.1, 1]

for g in gamma_values:

    model = SVC(
        kernel="rbf",
        C=1,
        gamma=g,
        class_weight="balanced"
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    print(f"Gamma = {g}")
    print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
    print(f"Fail Precision: {precision_score(y_test, predictions, pos_label=1, zero_division=0):.3f}")
    print(f"Fail Recall: {recall_score(y_test, predictions, pos_label=1, zero_division=0):.3f}")
    print(f"Fail F1: {f1_score(y_test, predictions, pos_label=1, zero_division=0):.3f}")
    print()

Gamma = 0.001
Accuracy: 0.860
Fail Precision: 0.129
Fail Recall: 0.190
Fail F1: 0.154

Gamma = 0.01
Accuracy: 0.933
Fail Precision: 0.000
Fail Recall: 0.000
Fail F1: 0.000

Gamma = 0.1
Accuracy: 0.933
Fail Precision: 0.000
Fail Recall: 0.000
Fail F1: 0.000

Gamma = 1
Accuracy: 0.933
Fail Precision: 0.000
Fail Recall: 0.000
Fail F1: 0.000



In [ ]:
## 14. C and Gamma Experiment

C_values = [0.1, 1, 10]
gamma_values = [0.001, 0.01, 0.1]

for c in C_values:
    for g in gamma_values:

        model = SVC(
            kernel="rbf",
            C=c,
            gamma=g,
            class_weight="balanced"
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        accuracy = accuracy_score(y_test, predictions)
        recall = recall_score(y_test, predictions, pos_label=1)
        f1 = f1_score(y_test, predictions, pos_label=1)

        print(
            f"C={c}, Gamma={g} | "
            f"Accuracy={accuracy:.3f} | "
            f"Fail Recall={recall:.3f} | "
            f"Fail F1={f1:.3f}"
        )

C=0.1, Gamma=0.001 | Accuracy=0.828 | Fail Recall=0.333 | Fail F1=0.206
C=0.1, Gamma=0.01 | Accuracy=0.080 | Fail Recall=1.000 | Fail F1=0.127
C=0.1, Gamma=0.1 | Accuracy=0.067 | Fail Recall=1.000 | Fail F1=0.125
C=1, Gamma=0.001 | Accuracy=0.860 | Fail Recall=0.190 | Fail F1=0.154
C=1, Gamma=0.01 | Accuracy=0.933 | Fail Recall=0.000 | Fail F1=0.000
C=1, Gamma=0.1 | Accuracy=0.933 | Fail Recall=0.000 | Fail F1=0.000
C=10, Gamma=0.001 | Accuracy=0.927 | Fail Recall=0.095 | Fail F1=0.148
C=10, Gamma=0.01 | Accuracy=0.933 | Fail Recall=0.000 | Fail F1=0.000
C=10, Gamma=0.1 | Accuracy=0.933 | Fail Recall=0.000 | Fail F1=0.000


In [ ]:
## 15. Prepare Raw Data for Final Model

X_raw = df.drop(columns=["Time", "Pass/Fail"]).copy()
y_raw = df["Pass/Fail"].copy()

# Remove constant features
constant_columns = X_raw.columns[X_raw.nunique() <= 1]
X_raw = X_raw.drop(columns=constant_columns)

# Train-test split
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw,
    y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw
)

print("X_train:", X_train_raw.shape)
print("X_test :", X_test_raw.shape)
print("y_train:", y_train_raw.shape)
print("y_test :", y_test_raw.shape)

X_train: (1253, 474)
X_test : (314, 474)
y_train: (1253,)
y_test : (314,)


## 16. Final SVM Pipeline and Hyperparameter Tuning

For the final model, preprocessing and SVM are combined into a single Pipeline.

GridSearchCV is used to find suitable values of C and gamma using 5-fold cross-validation.

The F1 score is used for tuning because the dataset is highly imbalanced between Pass and Fail classes.

In [ ]:
## 16. Final SVM Pipeline and Hyperparameter Tuning

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

final_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        class_weight="balanced"
    ))
])

param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": [0.001, 0.01, 0.1]
}

final_grid = GridSearchCV(
    estimator=final_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

final_grid.fit(X_train_raw, y_train_raw)

print("Best Parameters:", final_grid.best_params_)
print("Best CV F1:", final_grid.best_score_)

Best Parameters: {'svm__C': 0.1, 'svm__gamma': 0.001}
Best CV F1: 0.19822750123646188


## 17. Final Model Evaluation

The best model selected through cross-validation is evaluated on the unseen test set.

Accuracy, precision, recall, F1-score, and the confusion matrix are used to evaluate the final model.

In [ ]:
## 17. Final Model Evaluation

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

final_model = final_grid.best_estimator_

final_pred = final_model.predict(X_test_raw)

print("Final Test Accuracy:", accuracy_score(y_test_raw, final_pred))

print("\nClassification Report:")
print(classification_report(y_test_raw, final_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_raw, final_pred))

Final Test Accuracy: 0.8280254777070064

Classification Report:
              precision    recall  f1-score   support

          -1       0.95      0.86      0.90       293
           1       0.15      0.33      0.21        21

    accuracy                           0.83       314
   macro avg       0.55      0.60      0.55       314
weighted avg       0.89      0.83      0.86       314


Confusion Matrix:
[[253  40]
 [ 14   7]]


In [ ]:
## 18. Save Final Model

import joblib

joblib.dump(final_model, "../models/secom_svm_model.pkl")

print("Final model saved successfully!")

Final model saved successfully!


## Final Result

The final SVM model achieved 82.8% test accuracy.

However, because the dataset is highly imbalanced, the Fail class performance is more important than accuracy alone. The model achieved 33% recall and 21% F1-score for the Fail class.

This shows that the model can classify many samples correctly but still has difficulty detecting a significant portion of actual failures.

In [ ]:
## 19. Verify Saved Model

loaded_model = joblib.load("../models/secom_svm_model.pkl")

loaded_pred = loaded_model.predict(X_test_raw)

print("Loaded model prediction successful!")
print("Loaded model accuracy:", accuracy_score(y_test_raw, loaded_pred))

Loaded model prediction successful!
Loaded model accuracy: 0.8280254777070064
